# Maya v0.13C3 — Targeted Articulation Refinement

Fresh **Google Colab + T4 GPU**. Run top to bottom once.

Upload exactly:

1. `maya_native_hindi_v013c2_checkpoint.zip`
2. `Maya_Phonetic_Foundation_v0.13C1_2_PHONE_ALIGNED.zip`
3. `Maya_Curated_Timbre_Corpus_v0.11.zip`

This stage starts from **v0.13C2 step 2000**.

Training policy:

- Primary weak families, 6×: `k/kh`, `ch/chh`, `v/y`, `m/n`
- Secondary families, 3×: `g/gh`, `j/jh`, `t/th`, `d/dh`, `p/ph`
- Other aligned CV items, 1×
- v0.11 timbre rehearsal, 10×
- LR `1e-5`
- 1000 updates
- diagnostics at 250 / 500 / 750 / 1000

Do not add words or Husn yet.

In [ ]:
# 1. GPU + upload + identify packages
import subprocess, zipfile, shutil, json, re
from pathlib import Path
from google.colab import files

subprocess.run(["nvidia-smi"], check=True)

print("Upload all 3 ZIPs together:")
print("1) maya_native_hindi_v013c2_checkpoint.zip")
print("2) Maya_Phonetic_Foundation_v0.13C1_2_PHONE_ALIGNED.zip")
print("3) Maya_Curated_Timbre_Corpus_v0.11.zip")

u = files.upload()
zips = [Path(n) for n in u if n.lower().endswith(".zip")]
assert len(zips) == 3, zips

root = Path("/content/maya_v013c3")
if root.exists():
    shutil.rmtree(root)
root.mkdir()

packs = []
for z in zips:
    d = root / z.stem
    d.mkdir()
    with zipfile.ZipFile(z) as zz:
        zz.extractall(d)
    packs.append((z.name, d))

checkpoint_dir = None
aligned_dir = None
timbre_dir = None

for name, d in packs:
    if list(d.rglob("*.ckpt")):
        checkpoint_dir = d
        continue

    summaries = list(d.rglob("SUMMARY.json"))
    summary_text = " ".join(
        p.read_text(encoding="utf-8", errors="ignore")
        for p in summaries
    )

    if "PHONE ALIGNED" in summary_text:
        aligned_dir = d
    elif list(d.rglob("raw/transcriptions.csv")):
        timbre_dir = d

assert checkpoint_dir is not None, "C2 checkpoint ZIP not identified."
assert aligned_dir is not None, "C1.2 aligned corpus ZIP not identified."
assert timbre_dir is not None, "v0.11 timbre corpus ZIP not identified."

print("Checkpoint:", checkpoint_dir)
print("Aligned corpus:", aligned_dir)
print("Timbre corpus:", timbre_dir)

In [ ]:
# 2. Select C2 step-2000 checkpoint
from pathlib import Path
import re

ckpts = list(checkpoint_dir.rglob("*.ckpt"))
assert ckpts, "No checkpoint files found."

def step_of(p):
    m = re.search(r"(?:steps[_-]?|step[_-]?)(\d+)", p.name, re.I)
    if m:
        return int(m.group(1))
    nums = [int(x) for x in re.findall(r"\d+", p.stem)]
    return max(nums) if nums else -1

print("Available checkpoints:")
for p in sorted(ckpts, key=step_of):
    print(step_of(p), p)

step2000 = [p for p in ckpts if step_of(p) == 2000]
assert len(step2000) == 1, (
    "Could not uniquely identify step 2000. "
    f"Candidates: {step2000}"
)
base_ckpt = step2000[0]

configs = list(checkpoint_dir.rglob("config.yaml"))
assert configs, "No config.yaml in C2 package."
base_cfg = configs[0]

dicts = (
    list(checkpoint_dir.rglob("dictionary-mv.txt"))
    + list(checkpoint_dir.rglob("dictionary.txt"))
)
assert dicts, "No dictionary in C2 package."
base_dict = dicts[0]

print("✅ Base checkpoint:", base_ckpt)
print("✅ Base config:", base_cfg)
print("✅ Base dictionary:", base_dict)

In [ ]:
# 3. Fresh DiffSinger Python 3.10 environment
import subprocess, sys, shutil
from pathlib import Path

repo = Path("/content/DiffSinger")
venv = Path("/content/diffsinger-env")

for p in [repo, venv]:
    if p.exists():
        shutil.rmtree(p)

subprocess.run(
    ["git", "clone", "https://github.com/openvpi/DiffSinger.git", str(repo)],
    check=True
)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.10"], check=True)
subprocess.run(["uv", "venv", "--python", "3.10", "--seed", str(venv)], check=True)

PY = "/content/diffsinger-env/bin/python"

subprocess.run([PY, "-m", "pip", "install", "-U", "pip", "wheel"], check=True)
subprocess.run(
    [PY, "-m", "pip", "install", "--force-reinstall", "setuptools==81.0.0"],
    check=True
)
subprocess.run([
    PY, "-m", "pip", "install",
    "torch==2.4.0",
    "torchaudio==2.4.0",
    "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

subprocess.run(
    [PY, "-m", "pip", "install", "-r", str(repo / "requirements.txt")],
    check=True
)
subprocess.run(
    [PY, "-m", "pip", "install", "--force-reinstall", "setuptools==81.0.0"],
    check=True
)

subprocess.run([
    PY, "-c",
    "import sys,torch,pkg_resources;"
    "print(sys.version);print(torch.__version__);"
    "print(torch.cuda.is_available());"
    "assert sys.version_info[:2]==(3,10);"
    "assert torch.cuda.is_available()"
], check=True)

print("✅ DiffSinger environment ready.")

In [ ]:
# 4. Build targeted C3 dataset
import csv, shutil, soundfile as sf
from pathlib import Path

dst = Path("/content/DiffSinger/data/maya_v013c3")
raw = dst / "raw"
wavdir = raw / "wavs"
wavdir.mkdir(parents=True, exist_ok=True)

def one(base, pattern):
    xs = list(base.rglob(pattern))
    assert len(xs) == 1, (pattern, xs)
    return xs[0]

cv_csv = one(aligned_dir, "raw/transcriptions.csv")
tim_csv = one(timbre_dir, "raw/transcriptions.csv")

cv_raw = cv_csv.parent
tim_raw = tim_csv.parent

cv_rows = list(csv.DictReader(open(cv_csv, encoding="utf-8")))
tim_rows = list(csv.DictReader(open(tim_csv, encoding="utf-8")))

PRIMARY = {
    "z_k", "z_kh",
    "z_ch", "z_chh",
    "z_v", "z_y",
    "z_m", "z_n",
}

SECONDARY = {
    "z_g", "z_gh",
    "z_j", "z_jh",
    "z_t", "z_th",
    "z_d", "z_dh",
    "z_p", "z_ph",
}

VOWELS = {"a","e","i","o","u"}

def consonant_of(row):
    core = [
        p for p in row["ph_seq"].split()
        if p not in {"SP","AP"} | VOWELS
    ]
    assert len(core) == 1, (row["name"], core, row["ph_seq"])
    return core[0]

rows = []

for r in cv_rows:
    c = consonant_of(r)
    mult = 6 if c in PRIMARY else 3 if c in SECONDARY else 1

    for rep in range(mult):
        old = r["name"]
        new = f"cvx{rep+1:02d}_{old}"

        src = cv_raw / "wavs" / f"{old}.wav"
        assert src.exists(), src

        shutil.copy2(src, wavdir / f"{new}.wav")

        rr = dict(r)
        rr["name"] = new
        rows.append(rr)

# 10x v0.11 timbre rehearsal.
for rep in range(10):
    for r in tim_rows:
        old = r["name"]
        new = f"timr{rep+1:02d}_{old}"

        src = tim_raw / "wavs" / f"{old}.wav"
        assert src.exists(), src

        shutil.copy2(src, wavdir / f"{new}.wav")

        rr = dict(r)
        rr["name"] = new
        rows.append(rr)

with (raw / "transcriptions.csv").open(
    "w", newline="", encoding="utf-8"
) as f:
    w = csv.DictWriter(
        f,
        fieldnames=["name", "ph_seq", "ph_dur"]
    )
    w.writeheader()
    w.writerows(rows)

# Use exact C2 dictionary for checkpoint compatibility.
shutil.copy2(base_dict, dst / "dictionary.txt")

total = 0.0
for r in rows:
    wav = wavdir / f"{r['name']}.wav"
    d = sf.info(wav).duration
    pdur = sum(float(x) for x in r["ph_dur"].split())

    assert abs(d - pdur) < 0.025, (
        r["name"], d, pdur
    )
    total += d

print("Aligned CV source items:", len(cv_rows))
print("Effective targeted items:", sum(x["name"].startswith("cvx") for x in rows))
print("Timbre rehearsal items:", sum(x["name"].startswith("timr") for x in rows))
print("Total training items:", len(rows))
print("Effective audio:", round(total / 60, 2), "min")

In [ ]:
# 5. Build C3 config from C2 step-2000
from pathlib import Path
import shutil, yaml, json

pre = Path("/content/DiffSinger/checkpoints/maya_v013c2_step2000")
pre.mkdir(parents=True, exist_ok=True)

shutil.copy2(base_ckpt, pre / "pretrained.ckpt")
shutil.copy2(base_dict, pre / "dictionary-mv.txt")

# Preserve maps if C2 package contains them.
for name in ["lang_map.json", "spk_map.json"]:
    found = list(checkpoint_dir.rglob(name))
    if found:
        shutil.copy2(found[0], pre / name)

cfg = yaml.safe_load(
    base_cfg.read_text(encoding="utf-8")
)

cfg["base_config"] = []
cfg["dictionaries"] = {
    "mv": "data/maya_v013c3/dictionary.txt"
}
cfg["extra_phonemes"] = []
cfg["merged_phoneme_groups"] = []

names = [
    r["name"]
    for r in rows
    if r["name"].startswith("cvx")
]

cfg["datasets"] = [{
    "raw_data_dir": "data/maya_v013c3/raw",
    "speaker": "maya",
    "spk_id": 0,
    "language": "mv",
    "test_prefixes": [
        names[-1],
        names[-15],
        names[-40],
        names[-80],
    ]
}]

cfg["binary_data_dir"] = "data/maya_v013c3/binary"
cfg["binarization_args"] = {"num_workers": 0}

cfg["finetune_enabled"] = True
cfg["finetune_ckpt_path"] = (
    "checkpoints/maya_v013c2_step2000/pretrained.ckpt"
)
cfg["finetune_ignored_params"] = []
cfg["finetune_strict_shapes"] = True

cfg["freezing_enabled"] = False
cfg["frozen_params"] = []

cfg["optimizer_args"]["lr"] = 0.00001
cfg["lr_scheduler_args"]["step_size"] = 500
cfg["lr_scheduler_args"]["gamma"] = 0.85

cfg["max_batch_frames"] = 22000
cfg["max_batch_size"] = 12
cfg["max_updates"] = 1000
cfg["val_check_interval"] = 250
cfg["num_valid_plots"] = 4
cfg["val_with_vocoder"] = False
cfg["num_ckpt_keep"] = 8
cfg["permanent_ckpt_start"] = 250
cfg["permanent_ckpt_interval"] = 250
cfg["pl_trainer_devices"] = "auto"
cfg["pl_trainer_precision"] = "16-mixed"

cfg_path = dst / "maya_v013c3.yaml"
cfg_path.write_text(
    yaml.safe_dump(
        cfg,
        sort_keys=False,
        allow_unicode=True
    ),
    encoding="utf-8"
)

print("✅ Config:", cfg_path)

In [ ]:
# 6. Install R&D vocoder + binarize + train
import requests, zipfile, io, os, subprocess

url = (
    "https://github.com/openvpi/vocoders/releases/download/"
    "pc-nsf-hifigan-44.1k-hop512-128bin-2025.02/"
    "pc_nsf_hifigan_44.1k_hop512_128bin_2025.02.zip"
)

r = requests.get(url, timeout=180)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall("/content/DiffSinger/checkpoints")

PY = "/content/diffsinger-env/bin/python"
REPO = "/content/DiffSinger"
CFG = "data/maya_v013c3/maya_v013c3.yaml"
EXP = "maya_native_hindi_v013c3"

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["PYTHONUNBUFFERED"] = "1"

p = subprocess.run(
    [PY, "scripts/binarize.py", "--config", CFG],
    cwd=REPO,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env
)

print(p.stdout)

if p.returncode != 0:
    raise RuntimeError(
        "BINARIZATION FAILED. Full error is printed above."
    )

print("✅ BINARIZATION PASSED")

p = subprocess.run(
    [
        PY,
        "scripts/train.py",
        "--config", CFG,
        "--exp_name", EXP,
        "--reset"
    ],
    cwd=REPO,
    text=True,
    env=env
)

if p.returncode != 0:
    raise RuntimeError(
        f"TRAINING FAILED: {p.returncode}"
    )

print("✅ TRAINING COMPLETE")

In [ ]:
# 7. Prepare inference metadata + diagnostic DS files
from pathlib import Path
import shutil, json, yaml, numpy as np

REPO = Path("/content/DiffSinger")
EXP = "maya_native_hindi_v013c3"
exp_dir = REPO / "checkpoints" / EXP

assert exp_dir.exists(), exp_dir

shutil.copy2(
    REPO / "data/maya_v013c3/dictionary.txt",
    exp_dir / "dictionary-mv.txt"
)

if not (exp_dir / "lang_map.json").exists():
    (exp_dir / "lang_map.json").write_text(
        json.dumps({"mv": 0}, indent=2),
        encoding="utf-8"
    )

cfg_final = yaml.safe_load(
    (exp_dir / "config.yaml").read_text(
        encoding="utf-8"
    )
)

if cfg_final.get("use_spk_id", False):
    if not (exp_dir / "spk_map.json").exists():
        (exp_dir / "spk_map.json").write_text(
            json.dumps({"maya": 0}, indent=2),
            encoding="utf-8"
        )

tests = REPO / "data" / "maya_v013c3_tests"
tests.mkdir(parents=True, exist_ok=True)

def make_ds(name, seq, durs, freq=230.0):
    duration = float(sum(durs))
    dt = 0.01

    t = np.arange(0, duration, dt)

    midi = (
        69
        + 12 * np.log2(freq / 440.0)
        + 0.08 * np.sin(2 * np.pi * 5.0 * t)
    )

    f0 = 440 * 2 ** ((midi - 69) / 12)

    ds = [{
        "offset": 0.0,
        "ph_seq": " ".join(seq),
        "ph_dur": " ".join(
            f"{x:.6f}" for x in durs
        ),
        "f0_seq": " ".join(
            f"{x:.4f}" for x in f0
        ),
        "f0_timestep": dt,
        "lang": "mv"
    }]

    p = tests / f"{name}.ds"
    p.write_text(
        json.dumps(
            ds,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    return p

vowel_tests = [
    make_ds(
        f"vowel_{v}",
        [v],
        [1.6],
        240.0
    )
    for v in ["a","e","i","o","u"]
]

ALL_CV = [
    "z_k","z_kh","z_g","z_gh",
    "z_ch","z_chh","z_j","z_jh",
    "z_T","z_Th","z_D","z_Dh",
    "z_t","z_th","z_d","z_dh",
    "z_p","z_ph","z_b","z_bh",
    "z_s","z_sh","z_h",
    "z_r","z_l","z_v","z_y",
    "z_m","z_n"
]

cv_tests = [
    make_ds(
        f"cv_{i:02d}_{p.replace('z_','')}",
        [p, "a"],
        [0.24, 0.90],
        230.0
    )
    for i,p in enumerate(ALL_CV, 1)
]

print("Vowels:", len(vowel_tests))
print("CV:", len(cv_tests))

In [ ]:
# 8. Render steps 250/500/750/1000 + metrics + exports
import os, subprocess, shutil, json
from pathlib import Path

import soundfile as sf
import numpy as np
import librosa
import pandas as pd

from google.colab import files
from IPython.display import Audio, display

PY = "/content/diffsinger-env/bin/python"
REPO = Path("/content/DiffSinger")
EXP = "maya_native_hindi_v013c3"

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["PYTHONUNBUFFERED"] = "1"

steps = [250, 500, 750, 1000]

def onset_feature(path, seconds=.35):
    y, sr = librosa.load(
        path,
        sr=22050,
        mono=True
    )

    y = y[:int(seconds * sr)]

    mf = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=20,
        n_fft=1024,
        hop_length=256
    )

    return np.mean(mf, axis=1)

def cosine(a, b):
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / den) if den > 0 else None

def montage(paths, out):
    parts = []
    sr0 = None

    for p in paths:
        y, sr = sf.read(p)

        if y.ndim > 1:
            y = y.mean(axis=1)

        sr0 = sr0 or sr

        peak = np.max(np.abs(y)) + 1e-9
        if peak > .9:
            y *= .9 / peak

        parts += [
            y,
            np.zeros(int(.45 * sr))
        ]

    sf.write(
        out,
        np.concatenate(parts),
        sr0
    )

pairs = {
    "k_kh": ("z_k", "z_kh"),
    "ch_chh": ("z_ch", "z_chh"),
    "v_y": ("z_v", "z_y"),
    "m_n": ("z_m", "z_n"),
    "g_gh": ("z_g", "z_gh"),
    "j_jh": ("z_j", "z_jh"),
    "t_th": ("z_t", "z_th"),
    "d_dh": ("z_d", "z_dh"),
    "p_ph": ("z_p", "z_ph"),
}

results = []

for step in steps:
    print("\n======================")
    print("STEP", step)
    print("======================")

    outroot = Path(
        f"/content/maya_v013c3_step_{step}"
    )

    if outroot.exists():
        shutil.rmtree(outroot)

    outroot.mkdir()

    for ds in vowel_tests + cv_tests:
        p = subprocess.run(
            [
                PY,
                "scripts/infer.py",
                "acoustic",
                str(ds.relative_to(REPO)),
                "--exp", EXP,
                "--ckpt", str(step),
                "--out", str(outroot / ds.stem)
            ],
            cwd=str(REPO),
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env
        )

        if p.returncode != 0:
            print(p.stdout)
            raise RuntimeError(
                f"Inference failed step={step}: {ds.name}"
            )

    wavs = sorted(outroot.rglob("*.wav"))

    vowel_wavs = [
        p for p in wavs
        if p.stem.startswith("vowel_")
    ]

    cv_wavs = [
        p for p in wavs
        if p.stem.startswith("cv_")
    ]

    assert len(vowel_wavs) == 5
    assert len(cv_wavs) == 29

    phone_to_wav = {}

    for i, phone in enumerate(ALL_CV, 1):
        matches = [
            p for p in cv_wavs
            if p.stem.startswith(f"cv_{i:02d}_")
        ]
        assert len(matches) == 1, (phone, matches)
        phone_to_wav[phone] = matches[0]

    row = {"step": step}

    for label, (a, b) in pairs.items():
        row[label + "_cosine"] = cosine(
            onset_feature(phone_to_wav[a]),
            onset_feature(phone_to_wav[b])
        )

    row["target_pair_mean_cosine"] = float(
        np.mean([
            row[k + "_cosine"]
            for k in pairs
        ])
    )

    # Global CV separation for regression tracking.
    feats = [
        onset_feature(phone_to_wav[p])
        for p in ALL_CV
    ]

    vals = []

    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            vals.append(
                cosine(
                    feats[i],
                    feats[j]
                )
            )

    row["all_cv_mean_cosine"] = float(
        np.mean(vals)
    )

    results.append(row)

    diag = Path(
        f"/content/Maya_v013C3_DIAG_step_{step}.wav"
    )

    montage(
        vowel_wavs + cv_wavs,
        diag
    )

df = pd.DataFrame(results)

print("\nCHECKPOINT METRICS")
display(df)

metrics = Path(
    "/content/Maya_v013C3_metrics.csv"
)

df.to_csv(
    metrics,
    index=False
)

summary = {
    "model": "Maya Native Hindi v0.13C3",
    "base": "v0.13C2 step 2000",
    "learning_rate": 1e-5,
    "max_updates": 1000,
    "primary_oversampling": 6,
    "secondary_oversampling": 3,
    "timbre_rehearsal_multiplier": 10,
    "diagnostic_steps": steps,
    "status": "PENDING_CHECKPOINT_SELECTION"
}

summary_path = Path(
    "/content/Maya_v013C3_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)

ckzip = shutil.make_archive(
    "/content/maya_native_hindi_v013c3_checkpoint",
    "zip",
    REPO / "checkpoints" / EXP
)

for step in steps:
    files.download(
        f"/content/Maya_v013C3_DIAG_step_{step}.wav"
    )

files.download(str(metrics))
files.download(str(summary_path))
files.download(ckzip)

print("✅ C3 EXPORT COMPLETE")

## Upload back to ChatGPT

- `Maya_v013C3_DIAG_step_250.wav`
- `Maya_v013C3_DIAG_step_500.wav`
- `Maya_v013C3_DIAG_step_750.wav`
- `Maya_v013C3_DIAG_step_1000.wav`
- `Maya_v013C3_metrics.csv`
- `Maya_v013C3_summary.json`
- `maya_native_hindi_v013c3_checkpoint.zip`

We will select the best checkpoint before introducing isolated Hindi words.